In [1]:
!pip uninstall -y torch torchvision torchaudio
!pip install -U --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 227.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 164.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 301.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 184.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 182.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 192.2 MB/s eta 0:00:00a 0:00:01
    

In [2]:
import os
import random
import pickle
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from tqdm import tqdm


# =========================================================
# SETTINGS
# =========================================================
BASE_DIR = Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data")

FOLDERS = [
    "images_001",
    "images_002",
    "images_003",
    "images_004",
    "images_005",
    "images_006",
    "images_007",
    "images_008",
    "images_009",
    "images_010",
    "images_011",
    "images_012",
]

SAMPLE_SIZE = 250
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Path to save embeddings
SAVE_EMBEDDINGS_PATH = "/kaggle/working/chest_xray_reference_embeddings.pkl"

# Threshold for deciding "chest xray" vs "other"
# You may tune this later
SIMILARITY_THRESHOLD = 0.75


# =========================================================
# GET ALL IMAGE PATHS
# =========================================================
def get_all_image_paths(base_dir, folders):
    image_paths = []

    for folder in folders:
        folder_path = base_dir / folder / "images"
        if folder_path.exists():
            for ext in ["*.png", "*.jpg", "*.jpeg", "*.bmp"]:
                image_paths.extend(folder_path.glob(ext))

    return image_paths


all_image_paths = get_all_image_paths(BASE_DIR, FOLDERS)
print(f"Total images found: {len(all_image_paths)}")

if len(all_image_paths) < SAMPLE_SIZE:
    raise ValueError(f"Only {len(all_image_paths)} images found, less than SAMPLE_SIZE={SAMPLE_SIZE}")


# =========================================================
# PICK 250 RANDOM IMAGES
# =========================================================
random.seed(42)
sampled_paths = random.sample(all_image_paths, SAMPLE_SIZE)
print(f"Randomly selected {len(sampled_paths)} images.")


# =========================================================
# DATASET
# =========================================================
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # chest xray is grayscale, model expects 3 channels
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class ImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("L")  # grayscale
        if self.transform:
            image = self.transform(image)
        return image, str(img_path)


dataset = ImageDataset(sampled_paths, transform=transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


# =========================================================
# LOAD EFFICIENTNET-B3 AS FEATURE EXTRACTOR
# =========================================================
weights = EfficientNet_B3_Weights.DEFAULT
model = efficientnet_b3(weights=weights)

# Remove classifier, keep feature extractor
model.classifier = torch.nn.Identity()
model = model.to(DEVICE)
model.eval()


# =========================================================
# CREATE REFERENCE EMBEDDINGS
# =========================================================
all_embeddings = []
all_paths = []

with torch.no_grad():
    for images, paths in tqdm(loader, desc="Creating embeddings"):
        images = images.to(DEVICE)

        feats = model(images)                  # shape: [B, 1536]
        feats = F.normalize(feats, p=2, dim=1) # L2 normalize

        all_embeddings.append(feats.cpu().numpy())
        all_paths.extend(paths)

reference_embeddings = np.concatenate(all_embeddings, axis=0)
print("Reference embeddings shape:", reference_embeddings.shape)


# =========================================================
# SAVE REFERENCE EMBEDDINGS
# =========================================================
with open(SAVE_EMBEDDINGS_PATH, "wb") as f:
    pickle.dump({
        "embeddings": reference_embeddings,
        "paths": all_paths
    }, f)

print(f"Saved reference embeddings to: {SAVE_EMBEDDINGS_PATH}")


# =========================================================
# SINGLE IMAGE EMBEDDING FUNCTION
# =========================================================
def get_image_embedding(image_path, model, transform, device):
    image = Image.open(image_path).convert("L")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = model(image)
        feat = F.normalize(feat, p=2, dim=1)

    return feat.cpu().numpy()[0]


# =========================================================
# COSINE SIMILARITY CHECK
# =========================================================
def cosine_similarity_numpy(vec, matrix):
    # vec shape: [D]
    # matrix shape: [N, D]
    return np.dot(matrix, vec)


def detect_if_chest_xray(
    input_image_path,
    reference_embeddings,
    model,
    transform,
    device,
    threshold=0.75
):
    query_embedding = get_image_embedding(input_image_path, model, transform, device)

    similarities = cosine_similarity_numpy(query_embedding, reference_embeddings)

    max_similarity = similarities.max()
    mean_top5_similarity = np.mean(np.sort(similarities)[-5:])

    prediction = "CHEST_XRAY" if max_similarity >= threshold else "OTHER_IMAGE"

    return {
        "prediction": prediction,
        "max_similarity": float(max_similarity),
        "mean_top5_similarity": float(mean_top5_similarity)
    }


# =========================================================
# TEST ON NEW IMAGE
# =========================================================
# Change this path to your test image
TEST_IMAGE_PATH = "/kaggle/input/your-test-image-folder/test_image.png"

if os.path.exists(TEST_IMAGE_PATH):
    result = detect_if_chest_xray(
        input_image_path=TEST_IMAGE_PATH,
        reference_embeddings=reference_embeddings,
        model=model,
        transform=transform,
        device=DEVICE,
        threshold=SIMILARITY_THRESHOLD
    )

    print("\nDetection Result:")
    print("Prediction        :", result["prediction"])
    print("Max Similarity    :", result["max_similarity"])
    print("Mean Top-5 Sim    :", result["mean_top5_similarity"])
else:
    print(f"Test image not found: {TEST_IMAGE_PATH}")

Total images found: 112120
Randomly selected 250 images.
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 170MB/s] 
Creating embeddings: 100%|██████████| 16/16 [00:05<00:00,  3.11it/s]

Reference embeddings shape: (250, 1536)
Saved reference embeddings to: /kaggle/working/chest_xray_reference_embeddings.pkl
Test image not found: /kaggle/input/your-test-image-folder/test_image.png


In [6]:
import os
import pickle
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_EMBEDDINGS_PATH = "/kaggle/working/chest_xray_reference_embeddings.pkl"
TEST_IMAGE_PATH = "/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_002/images/00001336_000.png"
SIMILARITY_THRESHOLD = 0.50

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

weights = EfficientNet_B3_Weights.DEFAULT
model = efficientnet_b3(weights=weights)
model.classifier = torch.nn.Identity()
model = model.to(DEVICE)
model.eval()

with open(SAVE_EMBEDDINGS_PATH, "rb") as f:
    data = pickle.load(f)

reference_embeddings = data["embeddings"]
reference_paths = data["paths"]


def get_image_embedding(image_path, model, transform, device):
    image = Image.open(image_path).convert("L")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = model(image)
        feat = F.normalize(feat, p=2, dim=1)

    return feat.cpu().numpy()[0]


def cosine_similarity_numpy(vec, matrix):
    return np.dot(matrix, vec)


def detect_other_image(input_image_path, threshold=0.50):
    query_embedding = get_image_embedding(input_image_path, model, transform, DEVICE)
    similarities = cosine_similarity_numpy(query_embedding, reference_embeddings)

    max_similarity = similarities.max()
    top5_similarities = np.sort(similarities)[-5:]
    mean_top5_similarity = np.mean(top5_similarities)

    # Better logic:
    # if mean top-5 similarity is high => image is NOT_OTHER
    # else => image is OTHER
    if mean_top5_similarity >= threshold:
        prediction = "NOT_OTHER"
    else:
        prediction = "OTHER"

    return prediction, float(max_similarity), float(mean_top5_similarity), top5_similarities


if os.path.exists(TEST_IMAGE_PATH):
    prediction, max_sim, mean_top5, top5_vals = detect_other_image(
        TEST_IMAGE_PATH,
        SIMILARITY_THRESHOLD
    )

    print("Prediction            :", prediction)
    print("Max Similarity        :", max_sim)
    print("Mean Top-5 Similarity :", mean_top5)
    print("Top-5 Similarities    :", top5_vals)

    if prediction == "NOT_OTHER":
        print("Input image is NOT other image")
    else:
        print("Input image is OTHER image")
else:
    print("Test image not found.")

Prediction            : NOT_OTHER
Max Similarity        : 0.7590064406394958
Mean Top-5 Similarity : 0.7392336130142212
Top-5 Similarities    : [0.72947377 0.7295848  0.737049   0.74105406 0.75900644]
Input image is NOT other image
